# Nacimientos en Chile entre 2020 y 2023

## Avance Fase 3 – Semana 2

**MCDI500 – Programación para la Ciencia de Datos · Grupo 2**  
**Integrantes:** Cristian Hurtado, Jorge Altamirano y Renzo Vílchez  
**Docente:** Dr. Omar Salinas Silva

Este avance compara dos maneras de calcular la proporción regional de nacidos vivos de madres menores de 20 años y dos maneras de leer el CSV original. Parte de la salida de F2, verifica equivalencia y mide tiempo y memoria. Una demostración independiente codifica la región nominal con one-hot usando exclusivamente las categorías del conjunto de entrenamiento. No modifica F1, F2 ni los CSV de origen.

## 1. Datos, arquitectura y criterios

`data/processed/nacimientos_preparados_propuesta.csv` es la salida de F2. El módulo `F3/src/f3_algoritmos.py` separa lectura, cálculo y medición; el notebook coordina y documenta el experimento. `src/f2_utilidades.py` se reutiliza para aplanar metadatos anidados mediante recursión. La recursión es pertinente para una estructura de profundidad variable, pero no para recorrer filas tabulares.

El indicador es `100 × nacidos vivos de madres menores de 20 años / todos los nacidos vivos de la región`. La unidad es el nacido vivo inscrito, no la madre ni el embarazo. El grupo etario y la marca `madre_menor_20` provienen de F2. La región se mantiene como categoría para tablas legibles; la codificación se realiza en una matriz separada, siguiendo la indicación del docente. El escalamiento de talla no se aplica aquí: este prototipo no usa distancias y la talla debe conservar su unidad en centímetros.

La discusión del foro técnico de la Semana 1 se refiere a comparar alternativas que producen el mismo resultado y medir su costo temporal y espacial. Este cuaderno aplica ese criterio; la intervención concreta del grupo se incorporará cuando esté disponible (Universidad Andrés Bello, 2026).

## 2. Preparación del entorno

Se localiza la raíz del repositorio por la presencia de la salida preparada y el módulo F2. No se depende de que VS Code abra el notebook desde una carpeta concreta. Se imprimen las versiones usadas para la reproducibilidad (Salinas Silva, 2026).

In [1]:
from pathlib import Path
import gc
import json
import platform
import sys
from datetime import date

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

def encontrar_raiz(inicio=Path.cwd()):
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'data/processed/nacimientos_preparados_propuesta.csv').is_file() and (candidato / 'src/f2_utilidades.py').is_file():
            return candidato
    raise FileNotFoundError('Abra el repositorio proyecto-grupo2-f3-cristian y compruebe la salida de F2.')

RAIZ = encontrar_raiz()
sys.dont_write_bytecode = True
sys.path.insert(0, str(RAIZ / 'F3/src'))
sys.path.insert(0, str(RAIZ / 'src'))
import f3_algoritmos as f3
from f2_utilidades import aplanar

PREPARADO = RAIZ / 'data/processed/nacimientos_preparados_propuesta.csv'
ORIGINAL = RAIZ / 'data/raw/Serie_Nacimientos_2020_2023.csv'
RESULTADOS = RAIZ / 'F3/resultados'
RESULTADOS.mkdir(parents=True, exist_ok=True)
VERSIONES = {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__}
print('Fecha de ejecución:', date.today().isoformat())
print('Raíz:', RAIZ)
print('Versiones:', VERSIONES)

Fecha de ejecución: 2026-09-28
Raíz: D:\Magister UNAB\Curso 01\C01Week01\Evaluaciones\proyecto-grupo2-f3-cristian
Versiones: {'python': '3.13.5', 'pandas': '2.2.3', 'numpy': '2.1.3', 'scikit-learn': '1.6.1'}


## 3. Entrada preparada de F2 y verificaciones iniciales

Se leen solo las columnas del indicador, sin rehacer limpieza ni imputación. Se revisan tamaño, tipos y completitud antes de medir. El CSV completo de F2 permanece disponible como versión interpretable para tablas y figuras.

In [2]:
datos = f3.cargar_preparado(PREPARADO)
assert len(datos) > 0
assert datos.REGION_RESIDENCIA.between(1, 16).all()
assert datos.madre_menor_20.dtype == bool
print('Filas del conjunto preparado:', f'{len(datos):,}')
print('Regiones:', datos.REGION_RESIDENCIA.nunique())
print('Nulos en columnas usadas:', datos.isna().sum().to_dict())
display(datos.head())

Filas del conjunto preparado: 735,611
Regiones: 16
Nulos en columnas usadas: {'REGION_RESIDENCIA': 0, 'madre_menor_20': 0}


,REGION_RESIDENCIA,madre_menor_20
0,13,False
1,13,False
2,13,False
3,13,True
4,1,False


## 4. Dos implementaciones del mismo indicador

La primera usa un bucle y un diccionario de acumuladores. La segunda delega el agrupamiento a pandas. Ambas recorren conceptualmente las filas una vez; la comparación empírica revela el costo de ejecutar la iteración en Python frente a las operaciones de biblioteca. Antes de hablar de velocidad se exige igualdad de resultados.

In [3]:
resultado_bucle = f3.proporcion_bucle(datos)
resultado_groupby = f3.proporcion_groupby(datos)
pd.testing.assert_frame_equal(resultado_bucle, resultado_groupby, check_dtype=False, rtol=0, atol=1e-12)
assert int(resultado_bucle.nacimientos.sum()) == len(datos)
assert int(resultado_bucle.madres_menores_20.sum()) == int(datos.madre_menor_20.sum())
print('Verificación: resultados equivalentes y sin pérdida de filas.')
display(resultado_groupby.head())

Verificación: resultados equivalentes y sin pérdida de filas.


,REGION_RESIDENCIA,nacimientos,madres_menores_20,porcentaje
0,1,18801,1198,6.372001
1,2,28659,1642,5.729439
2,3,12555,736,5.862206
3,4,31196,1710,5.481472
4,5,70037,3167,4.521896


## 5. Medición de las dos implementaciones

Cada función se ejecuta tres veces sobre las 735.611 filas y se informa el menor tiempo, siguiendo la guía docente. Los tiempos son mediciones del equipo que ejecuta el notebook, no constantes universales. Se conserva la equivalencia verificada en la sección anterior. Se usa `time.perf_counter()` para el cronometraje (Python Software Foundation, s. f.).

In [4]:
tiempos_bucle = f3.medir_tiempos(lambda: f3.proporcion_bucle(datos), repeticiones=3)
tiempos_groupby = f3.medir_tiempos(lambda: f3.proporcion_groupby(datos), repeticiones=3)
tabla_indicador = pd.DataFrame([
    {'implementacion': 'bucle Python', 'repeticiones_s': tiempos_bucle, 'mejor_s': min(tiempos_bucle)},
    {'implementacion': 'pandas groupby', 'repeticiones_s': tiempos_groupby, 'mejor_s': min(tiempos_groupby)},
])
display(tabla_indicador)
print('Razón de tiempos bucle/groupby:', round(min(tiempos_bucle) / min(tiempos_groupby), 2))

,implementacion,repeticiones_s,mejor_s
0,bucle Python,"[0.14589370001340285, 0.14611390000209212, 0.1...",0.144367
1,pandas groupby,"[0.009655799949541688, 0.009374000015668571, 0...",0.009238


Razón de tiempos bucle/groupby: 15.63


## 6. Lectura completa frente a lectura selectiva

El segundo experimento usa el CSV original **solo para comparar el costo de lectura**, sin sustituir al conjunto preparado de F2. La versión A lee las 25 columnas; la B pide las seis columnas originales que F2 utiliza. Se comprueba que las seis columnas coinciden fila por fila, se mide el tiempo tres veces y se compara la memoria ocupada por los DataFrames mediante `memory_usage(deep=True)`; esta última cifra no equivale al pico total del proceso. El argumento de `usecols` está documentado por pandas (The pandas development team, s. f.).

In [5]:
if not ORIGINAL.is_file():
    raise FileNotFoundError(f'Falta el CSV original para el experimento de lectura: {ORIGINAL}')
completo = f3.leer_original(ORIGINAL)
selectivo = f3.leer_original(ORIGINAL, f3.COLUMNAS_ORIGEN)
assert len(completo) == len(selectivo) == len(datos)
pd.testing.assert_frame_equal(completo[f3.COLUMNAS_ORIGEN], selectivo[f3.COLUMNAS_ORIGEN], check_dtype=False)
memoria_completa_mb = completo.memory_usage(index=True, deep=True).sum() / 1024**2
memoria_selectiva_mb = selectivo.memory_usage(index=True, deep=True).sum() / 1024**2
print('Columnas:', len(completo.columns), 'frente a', len(selectivo.columns))
print('Equivalencia de las seis columnas: comprobada.')
del completo, selectivo
gc.collect()
tiempos_completa = f3.medir_tiempos(lambda: f3.leer_original(ORIGINAL), repeticiones=3)
tiempos_selectiva = f3.medir_tiempos(lambda: f3.leer_original(ORIGINAL, f3.COLUMNAS_ORIGEN), repeticiones=3)
tabla_lectura = pd.DataFrame([
    {'lectura': '25 columnas', 'repeticiones_s': tiempos_completa, 'mejor_s': min(tiempos_completa), 'memoria_dataframe_mb': memoria_completa_mb},
    {'lectura': '6 columnas', 'repeticiones_s': tiempos_selectiva, 'mejor_s': min(tiempos_selectiva), 'memoria_dataframe_mb': memoria_selectiva_mb},
])
display(tabla_lectura)
print('Razón de tiempos completa/selectiva:', round(min(tiempos_completa) / min(tiempos_selectiva), 2))

Columnas: 25 frente a 6
Equivalencia de las seis columnas: comprobada.


,lectura,repeticiones_s,mejor_s,memoria_dataframe_mb
0,25 columnas,"[0.7546734000206925, 0.7514824999962002, 0.730...",0.730797,408.081716
1,6 columnas,"[0.5078645999892615, 0.49931079999078065, 0.49...",0.495111,127.295326


Razón de tiempos completa/selectiva: 1.48


## 7. Codificación nominal sin fuga de categorías

`REGION_RESIDENCIA` contiene códigos 1–16, pero sus números no implican orden ni distancia. Se trata como texto nominal. Se separan filas en entrenamiento (80 %) y prueba (20 %) con semilla fija; el vocabulario se aprende **solo del entrenamiento**. `handle_unknown='ignore'` conserva las mismas columnas ante una categoría nueva en prueba (scikit-learn developers, s. f.; Pedregosa et al., 2011). La matriz es dispersa para evitar 16 columnas densas en memoria. Los índices de fila conservan la correspondencia con el CSV preparado y, por tanto, con su versión legible para F4.

In [6]:
categorias_region = datos[['REGION_RESIDENCIA']].astype(str)
indices = np.arange(len(datos))
idx_entrenamiento, idx_prueba = train_test_split(indices, test_size=0.20, random_state=42, stratify=categorias_region.REGION_RESIDENCIA)
codificador = OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.uint8)
matriz_entrenamiento = codificador.fit_transform(categorias_region.iloc[idx_entrenamiento])
matriz_prueba = codificador.transform(categorias_region.iloc[idx_prueba])
nombres_columnas = codificador.get_feature_names_out(['REGION_RESIDENCIA']).tolist()
assert matriz_entrenamiento.shape[1] == matriz_prueba.shape[1] == len(nombres_columnas)
assert np.all(np.asarray(matriz_entrenamiento.sum(axis=1)).ravel() == 1)
assert np.all(np.asarray(matriz_prueba.sum(axis=1)).ravel() == 1)
desconocida = pd.DataFrame({'REGION_RESIDENCIA': ['99']})
matriz_desconocida = codificador.transform(desconocida)
assert matriz_desconocida.shape[1] == matriz_prueba.shape[1]
assert matriz_desconocida.nnz == 0
assert codificador.inverse_transform(matriz_entrenamiento[:5]).ravel().tolist() == categorias_region.iloc[idx_entrenamiento[:5], 0].tolist()
print('Entrenamiento:', matriz_entrenamiento.shape, '| Prueba:', matriz_prueba.shape)
print('Vocabulario aprendido solo en entrenamiento:', codificador.categories_[0].tolist())
print('Categoría nueva en prueba: mismo esquema y fila de ceros; por sí sola no se puede invertir.')
print('Ejemplo legible, fila de origen y nombre de columna codificada:')
display(pd.DataFrame({'fila_origen': idx_entrenamiento[:5], 'region_original': categorias_region.iloc[idx_entrenamiento[:5], 0].tolist(), 'columna_activa': [nombres_columnas[i] for i in matriz_entrenamiento[:5].argmax(axis=1).A.ravel()]}))
parametros_codificacion = {
    'variable': 'REGION_RESIDENCIA', 'semilla': 42, 'proporcion_prueba': 0.20,
    'vocabulario_entrenamiento': codificador.categories_[0].tolist(),
    'columnas_codificadas': nombres_columnas,
    'filas_entrenamiento': len(idx_entrenamiento), 'filas_prueba': len(idx_prueba),
    'categorias_desconocidas': 'mismo esquema; fila de ceros; conservar original para interpretación',
    'escalamiento_talla': 'no aplicado; centímetros y análisis descriptivo sin distancias',
}
(RESULTADOS / 'parametros_codificacion.json').write_text(json.dumps(parametros_codificacion, ensure_ascii=False, indent=2), encoding='utf-8')

Entrenamiento: (588488, 16) | Prueba: (147123, 16)
Vocabulario aprendido solo en entrenamiento: ['1', '10', '11', '12', '13', '14', '15', '16', '2', '3', '4', '5', '6', '7', '8', '9']
Categoría nueva en prueba: mismo esquema y fila de ceros; por sí sola no se puede invertir.
Ejemplo legible, fila de origen y nombre de columna codificada:


,fila_origen,region_original,columna_activa
0,654935,10,REGION_RESIDENCIA_10
1,199523,13,REGION_RESIDENCIA_13
2,355966,9,REGION_RESIDENCIA_9
3,63411,13,REGION_RESIDENCIA_13
4,340687,13,REGION_RESIDENCIA_13


991

## 7.1. Ejemplo visible de la codificación one-hot

La tabla siguiente muestra cuatro regiones que sí aparecieron en entrenamiento. Cada **columna** representa una región original y cada **fila** una de las 16 columnas indicadoras creadas por el codificador. En cada ejemplo hay un único `1`: la fila que corresponde a su región. Los demás valores son `0`. Se muestran cuatro ejemplos para que la tabla sea legible; la matriz real incluye todas las filas del conjunto.

In [7]:
regiones_ejemplo = ['1', '5', '13', '16']
indices_ejemplo = [
    int(idx_entrenamiento[np.flatnonzero(
        categorias_region.iloc[idx_entrenamiento, 0].to_numpy() == region
    )[0]])
    for region in regiones_ejemplo
]
muestra_original = categorias_region.iloc[indices_ejemplo]
matriz_ejemplo = codificador.transform(muestra_original).toarray()
assert np.all(matriz_ejemplo.sum(axis=1) == 1)
tabla_onehot = pd.DataFrame(
    matriz_ejemplo.T,
    index=nombres_columnas,
    columns=[f'Región original {region}' for region in regiones_ejemplo],
)
tabla_onehot.index.name = 'Columna creada por one-hot'
print('Filas originales del CSV preparado:', indices_ejemplo)
display(tabla_onehot)

Filas originales del CSV preparado: [20448, 248638, 199523, 418577]


,Región original 1,Región original 5,Región original 13,Región original 16
Columna creada por one-hot,,,,
REGION_RESIDENCIA_1,1,0,0,0
REGION_RESIDENCIA_10,0,0,0,0
REGION_RESIDENCIA_11,0,0,0,0
REGION_RESIDENCIA_12,0,0,0,0
REGION_RESIDENCIA_13,0,0,1,0
REGION_RESIDENCIA_14,0,0,0,0
REGION_RESIDENCIA_15,0,0,0,0
REGION_RESIDENCIA_16,0,0,0,1
REGION_RESIDENCIA_2,0,0,0,0


## 8. Reutilización de recursividad de F2

La función `aplanar` de `src/f2_utilidades.py` convierte metadatos anidados en claves de ruta. Reutilizarla evita duplicar el algoritmo. El caso base es un valor no diccionario; el caso recursivo baja un nivel del diccionario. Esta decisión se aplica a metadatos, cuya profundidad puede variar, y no a los 735.611 registros tabulares (Salinas Silva, 2026).

In [8]:
ejemplo = {'fuente': {'institucion': 'DEIS/MINSAL'}, 'f3': {'medicion': {'repeticiones': 3}}}
esperado = {'fuente.institucion': 'DEIS/MINSAL', 'f3.medicion.repeticiones': 3}
assert aplanar(ejemplo) == esperado
display(pd.DataFrame(aplanar(ejemplo).items(), columns=['ruta', 'valor']))

,ruta,valor
0,fuente.institucion,DEIS/MINSAL
1,f3.medicion.repeticiones,3


## 9. Síntesis del experimento y trazabilidad

La elección entre las dos implementaciones del indicador debe basarse en equivalencia y tiempo medido. La lectura selectiva se justifica si reduce tiempo o memoria sin cambiar las seis columnas de interés; ambas dimensiones se muestran por separado. Los resultados dependen del equipo y del estado de sus archivos. F3 deja la matriz codificada separada del conjunto legible. La evolución hacia F4 podría encapsular el codificador como componente con estado (`fit`/`transform`) y separar lectura, cálculo y visualización, manteniendo una responsabilidad por módulo.

In [9]:
resumen = {
    'fecha': date.today().isoformat(), 'versiones': VERSIONES, 'filas': len(datos),
    'indicador': {'bucle_s': min(tiempos_bucle), 'groupby_s': min(tiempos_groupby), 'equivalencia': True},
    'lectura': {'completa_s': min(tiempos_completa), 'selectiva_s': min(tiempos_selectiva),
                'completa_mb': memoria_completa_mb, 'selectiva_mb': memoria_selectiva_mb, 'equivalencia': True},
    'codificacion': {'categorias_entrenamiento': len(codificador.categories_[0]), 'mismo_esquema': True},
}
(RESULTADOS / 'mediciones.json').write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding='utf-8')
print('Indicador: groupby', 'más rápido' if min(tiempos_groupby) < min(tiempos_bucle) else 'no más rápido', 'en esta ejecución.')
print('Lectura selectiva:', round(memoria_selectiva_mb, 2), 'MB frente a', round(memoria_completa_mb, 2), 'MB del DataFrame completo.')
print('Se guardaron solo metadatos y mediciones pequeñas en F3/resultados; no se duplicó el CSV.')

Indicador: groupby más rápido en esta ejecución.
Lectura selectiva: 127.3 MB frente a 408.08 MB del DataFrame completo.
Se guardaron solo metadatos y mediciones pequeñas en F3/resultados; no se duplicó el CSV.


## Referencias

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://www.jmlr.org/papers/v12/pedregosa11a.html

Python Software Foundation. (s. f.). *time — Time access and conversions*. https://docs.python.org/3/library/time.html

Salinas Silva, O. (2026). *Guía de apoyo: Evaluación Formativa 3, Fase 3* [Material de curso]. Universidad Andrés Bello.

Universidad Andrés Bello. (2026). *Formativa 2: Decisiones de diseño algorítmico* [Consigna de foro]. MCDI500.

scikit-learn developers. (s. f.). *OneHotEncoder*. https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

The pandas development team. (s. f.). *pandas.read_csv*. https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html